In [2]:
import os
import requests

OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")

def query_ollama(prompt: str):
    resp = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={"model": "tinyllama", "prompt": prompt},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()

In [5]:
from langchain_community.llms import Ollama

llm = Ollama(
    model="tinyllama",  # must match the model name you pulled with ollama
    temperature=0
)

print(llm.invoke("You are a travel agent. Suggest me a 3-day trip idea."))


C:\Users\OHB\AppData\Local\Temp\ipykernel_14024\106328713.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(


Sure, here's a 3-day trip idea for you:

Day 1: Arrival in Delhi
- Pick up from airport and transfer to your hotel
- Explore the city of Delhi, visiting iconic landmarks such as Red Fort, Jama Masjid, and Humayun's Tomb.
- Enjoy a delicious Indian breakfast at a local restaurant.

Day 2: Delhi - Agra (1 hour drive)
- Drive to Agra, where you will visit the iconic Taj Mahal and Agra Fort.
- Lunch at a local restaurant in Agra.
- Afternoon, explore the city of Agra, including the Agra Fort, Fatehpur Sikri, and Bahai House of Worship.

Day 3: Delhi - Jaipur (2 hours drive)
- Drive to Jaipur, where you will visit the City Palace, Hawa Mahal, and Jantar Mantar Observatory.
- Lunch at a local restaurant in Jaipur.
- Afternoon, explore the city of Jaipur, including the Amber Fort, Jal Mahal, and City Museum.

Enjoy your trip to Delhi, Agra, and Jaipur!


In [4]:
from typing import List, Dict
from langchain.tools import tool
from langchain_community.llms import Ollama

g:\My Drive\Project_Agent\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [6]:
@tool
def budget_calculator(user_budget: float, flight_price: float, hotel_price: float) -> dict:
    """Calculate total trip cost and check if it fits the user's budget."""
    total = flight_price + hotel_price
    return {
        "total_cost": total,
        "is_within_budget": total <= user_budget,
    }


@tool
def flight_search_api(origin: str, destination: str, date: str, max_price: float = None) -> List[Dict]:
    """
    Search for flights between origin and destination on a given date.
    Returns a list of flights with flightNumber, airline, departureTime, arrivalTime, and price.
    This is a MOCK – replace with real APIs later.
    """
    # mock data
    return [
        {
            "flightNumber": "TL101",
            "airline": "Tiny Air",
            "departureTime": f"{date}T08:00",
            "arrivalTime": f"{date}T10:30",
            "price": 200.0,
        },
        {
            "flightNumber": "TL202",
            "airline": "Tiny Air",
            "departureTime": f"{date}T16:00",
            "arrivalTime": f"{date}T18:30",
            "price": 260.0,
        },
    ]


@tool
def hotel_booking_api(city: str, check_in_date: str, check_out_date: str, max_price: float = None) -> List[Dict]:
    """
    Search for hotels in the given city between the given dates.
    Returns a list of hotels with hotelName, priceTotal, rating, and address.
    This is a MOCK – replace with real APIs later.
    """
    return [
        {
            "hotelName": "Tiny Budget Hotel",
            "priceTotal": 180.0,
            "rating": 3.9,
            "address": f"Downtown {city}",
        },
        {
            "hotelName": "Tiny Luxury Suites",
            "priceTotal": 350.0,
            "rating": 4.7,
            "address": f"Central {city}",
        },
    ]


In [7]:
llm = Ollama(
    model="tinyllama",
    temperature=0.2,  # a bit of creativity, but not too much
)

In [1]:
MEMORY_FILE = "memory.json"

def load_memory():
    if not os.path.exists(MEMORY_FILE):
        return {}
    with open(MEMORY_FILE, "r") as f:
        return json.load(f)

def save_memory(memory: dict):
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory, f, indent=4)

def update_memory(new_data: dict):
    memory = load_memory()
    for key, value in new_data.items():
        if value not in [None, "", "null"]:
            memory[key] = value
    save_memory(memory)


In [8]:
import json

def extract_trip_info(user_input: str, default_origin: str = "Istanbul") -> dict:
    """
    Ask tinyllama to extract structured trip info from the user's text.
    """
    memory = load_memory()

    prompt = f"""
You are an assistant that extracts structured trip information from user requests.

User request:
\"\"\"{user_input}\"\"\"

Extract these fields as JSON:
- origin (string, default "{default_origin}" if not provided)
- destination (string)
- depart_date (string, rough text is fine)
- return_date (string, if multi-day trip)
- total_budget (number, approximate if given as a range)
If something is missing, put null.

The memory has:
{json.dumps(memory, indent=2)}

If the user does not specify a field, reuse the value from memory.

Return ONLY JSON:
{{
  "origin": "...",
  "destination": "...",
  "depart_date": "...",
  "return_date": "...",
  "total_budget": number or null
}}
"""

    raw = llm.invoke(prompt)
    # Tinyllama might be messy; try to find JSON in the reply
    try:
        # attempt direct parse
        data = json.loads(raw)
    except Exception:
        # crude fallback: try to extract first {...} block
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1:
            data = json.loads(raw[start:end+1])
        else:
            data = {}

    # update memory
    update_memory(data)

    # combine memory + new data
    full = load_memory()
    return full


In [10]:
def choose_best_trip_option(flights, hotels, user_budget: float):
    valid_options = []

    for f in flights:
        for h in hotels:
            total_price = f["price"] + h["priceTotal"]
            if total_price <= user_budget:
                valid_options.append({
                    "flight": f,
                    "hotel": h,
                    "total_cost": total_price
                })

    if not valid_options:
        return None  # signal that nothing fits

    valid_options.sort(key=lambda x: x["total_cost"])
    return valid_options[0]

In [11]:
def tinyllama_travel_agent(user_input: str):
    # 1. Extract trip info using tinyllama
    trip = extract_trip_info(user_input)

    destination = trip.get("destination")
    depart_date = trip.get("depart_date")
    return_date = trip.get("return_date")
    total_budget = trip.get("total_budget") or 800  # default fallback
    origin = trip.get("origin") or "Istanbul"

    if not destination or not depart_date:
        return "I couldn't understand your destination or date. Please specify where and when you want to travel."

    # For simplicity, use depart_date as both for flights & hotels.
    # In a full version, you'd parse dates properly.
    check_in_date = depart_date
    check_out_date = return_date or depart_date

    # 2. Call tools: flights & hotels
    flights = flight_search_api.invoke({
        "origin": origin,
        "destination": destination,
        "date": depart_date,
        "max_price": None,
    })

    hotels = hotel_booking_api.invoke({
        "city": destination,
        "check_in_date": check_in_date,
        "check_out_date": check_out_date,
        "max_price": None,
    })

    # 3. Choose best option within budget
    best = choose_best_trip_option(flights, hotels, total_budget)

    if best is None:
        summary = f"I couldn't find a flight + hotel combination within your budget of {total_budget}."
        # Ask tinyllama to suggest alternatives textually
        alt_prompt = f"""
{summary}

Suggest in a friendly way how the user could adjust:
- budget
- dates
- destination or hotel type

Keep it under 5 sentences.
"""
        return llm.invoke(alt_prompt)

    # 4. Use tinyllama to nicely format the plan
    planning_prompt = f"""
You are a travel planning assistant.

Here is a chosen trip option:

Origin: {origin}
Destination: {destination}
Departure date: {depart_date}
Return/check-out date: {check_out_date}
Total budget: {total_budget}

Chosen flight:
- Airline: {best["flight"]["airline"]}
- Flight number: {best["flight"]["flightNumber"]}
- Departure time: {best["flight"]["departureTime"]}
- Arrival time: {best["flight"]["arrivalTime"]}
- Price: {best["flight"]["price"]}

Chosen hotel:
- Name: {best["hotel"]["hotelName"]}
- Address: {best["hotel"]["address"]}
- Rating: {best["hotel"]["rating"]}
- Price total: {best["hotel"]["priceTotal"]}

Total combined cost: {best["total_cost"]}

Write a clear, friendly 3–6 paragraph summary for the user:
- Confirm the plan
- Explain why this option fits the budget
- Suggest 2–3 simple activities for the destination
- Add 2 short travel tips
"""
    return llm.invoke(planning_prompt)


In [7]:
if __name__ == "__main__":
    print("💼 TinyLlama Travel Agent (local)")
    print("Type 'quit' to exit.\n")

    while True:
        user_input = input("You: ")
        if user_input.lower().strip() in ["quit", "exit"]:
            print("Agent: Safe travels, goodbye!")
            break

        response = tinyllama_travel_agent(user_input)
        print("\nAgent:\n", response, "\n")

💼 TinyLlama Travel Agent (local)
Type 'quit' to exit.


Agent:
 I couldn't understand your destination or date. Please specify where and when you want to travel. 


Agent:
 As a travel planning assistant, I'm happy to recommend a trip option that fits within your budget and includes two simple activities in Berlin. Here's how it works:

Origin: Istanbul
Destination: Berlin
Departure date: June 1st
Return/check-out date: June 1st
Total budget: $800

Choosen flight:
- Airlines: Tiyn Air
- Flight number: TL101
- Departure time: 2pm
- Arrival time: 4pm
- Price: $200.0

Choosen hotel:
- Name: Tiyn Budget Hotel
- Address: Downtown Berlin
- Rating: 3.9/5 stars
- Price total: $180.0

Total combined cost: $380.0

To make the most of your trip, here are two simple activities you can do in Berlin:

1. Visit the Berlin Wall Memorial and Museum
This is a must-visit attraction that tells the story of the Berlin Wall's construction and its eventual fall. You'll learn about the history of the wall and